In [54]:
import argparse
import os
import yaml
from transformers import AutoTokenizer, AutoModel
import torch

from scripts.GenaLMWithExtraFeatures import GenaLMWithExtraFeatures
from sklearn.discriminant_analysis import StandardScaler

In [55]:
os.chdir('/mnt/mr01-home01/m65338lb/worktrees/rnaDecay/gena_lm_extraFeatures')

In [56]:

parser = argparse.ArgumentParser()

# BASIC
parser.add_argument("--params", default='params.yaml', type=str, help="Path to the YAML file containing parameters.",)
parser.add_argument("--data_dir", default="output/data/decay", type=str, help="The input data dir. Should contain the .tsv files (or other data files) for the task.",)
parser.add_argument("--extraFeatures", default=None, type=str, help="Path the the csv file containing the extra features",)
parser.add_argument("--should_continue", action="store_true", help="Whether to continue from latest checkpoint in output_dir")
parser.add_argument("--config_name", default="", type=str, help="Pretrained config name or path if not the same as model_name",)
parser.add_argument("--model_name_or_path", default=None, type=str, help="Path to pre-trained model or shortcut name selected in the list",)
parser.add_argument("--task_name", default='rnaprom', type=str, help="Script only prepared for promoter task" )
parser.add_argument("--output_dir", default=None, type=str, help="The output directory where the model predictions and checkpoints will be written.",)
parser.add_argument("--tokenizer_name",default="rna3",type=str, help="Pretrained tokenizer name or path if not the same as model_name",)

# OBJECTIVE
parser.add_argument("--do_train", action="store_true", help="Whether to run training.")
parser.add_argument("--do_eval", action="store_true", help="Whether to run eval on the dev set.")
parser.add_argument("--do_predict", action="store_true", help="Whether to do prediction on the given dataset.")
parser.add_argument("--do_visualize", action="store_true", help="Whether to calculate attention score.")

# VALIDATION DURING TRAINING / EVALUATE 
parser.add_argument("--evaluate_during_training", action="store_true", help="Run evaluation during training at each logging step.",)
parser.add_argument("--do_visualize_during_training", action="store_true", help="Steps to generate an image")
parser.add_argument("--image_steps", type=int, default=0, help="Steps to generate an image")

# MODEL CONFIGS (only use)

# TRAINING DETAILS
parser.add_argument("--max_seq_length", default=512, type=int, help="The maximum total input sequence length after tokenization. Sequences longer "
                    "than this will be truncated, sequences shorter will be padded.",)
parser.add_argument("--per_gpu_train_batch_size", default=8, type=int, help="Batch size per GPU/CPU for training.",)
parser.add_argument("--per_gpu_eval_batch_size", default=8, type=int, help="Batch size per GPU/CPU for evaluation.",)
parser.add_argument("--per_gpu_pred_batch_size", default=8, type=int, help="Batch size per GPU/CPU for prediction.",)
parser.add_argument("--numEpochsBeforeEarlyStopping", default=10, type=int, help="Number of epochs to wait before tracking validation early stopping.",)
parser.add_argument("--patience", default=5, type=int, help="Number of epochs to wait before validation early stopping is triggered.",)
parser.add_argument("--learning_rate", default=5e-5, type=float, help="The initial learning rate for Adam.")
parser.add_argument("--gradient_accumulation_steps", type=int, default=1, help="Number of updates steps to accumulate before performing a backward/update pass.",)
parser.add_argument("--weight_decay", default=0.0, type=float, help="Weight decay if we apply some.")
parser.add_argument("--adam_epsilon", default=1e-8, type=float, help="Epsilon for Adam optimizer.")
parser.add_argument("--beta1", default=0.9, type=float, help="Beta1 for Adam optimizer.")
parser.add_argument("--beta2", default=0.999, type=float, help="Beta2 for Adam optimizer.")
parser.add_argument("--max_grad_norm", default=1.0, type=float, help="Max gradient norm.")
parser.add_argument("--attention_probs_dropout_prob", default=0.1, type=float, help="Dropout rate of attention.")
parser.add_argument("--hidden_dropout_prob", default=0.1, type=float, help="Dropout rate of intermidiete layer.")
parser.add_argument("--num_train_epochs", default=3.0, type=float, help="Total number of training epochs to perform.",)
parser.add_argument("--max_steps", default=-1, type=int, help="If > 0: set total number of training steps to perform. Override num_train_epochs.",)
parser.add_argument("--warmup_steps", default=0, type=int, help="Linear warmup over warmup_steps.")
parser.add_argument("--warmup_percent", default=0, type=float, help="Linear warmup over warmup_percent*total_steps.")
parser.add_argument("--seed", type=int, default=42, help="random seed for initialization")
parser.add_argument("--local_rank", type=int, default=-1, help="For distributed training: local_rank")
parser.add_argument("--n_process", default=2, type=int, help="number of processes used for data process",)
parser.add_argument("--eval_all_checkpoints", action="store_true", help="Evaluate all checkpoints starting with the same prefix as model_name ending and ending with step number",)
parser.add_argument("--no_cuda", action="store_true", help="Avoid using CUDA when available")
parser.add_argument("--logging_steps", type=int, default=500, help="Log every X updates steps.")
parser.add_argument("--save_steps", type=int, default=500, help="Save checkpoint every X updates steps.")
parser.add_argument("--save_total_limit", type=int, default=None, help="Limit the total amount of checkpoints, delete the older checkpoints in the output_dir, does not delete by default",)
parser.add_argument("--overwrite_output_dir", action="store_true", help="Overwrite the content of the output directory",)
parser.add_argument("--neptune", default=False, help="Neptune")
parser.add_argument("--neptune_tags", type=list, default=["trial"], help="Neptune tags")
parser.add_argument("--neptune_description", type=str, default="TRIAL minilm fine-tuning", help="Neptune description")
parser.add_argument("--neptune_token", type=str, default=None, help="Neptune API token")
parser.add_argument("--neptune_project", type=str, default=None, help="Neptune project")
parser.add_argument("--memory_size", type=int, default=None, help="number of memory tokens to use in RMT.",)
parser.add_argument("--block_size", type=int, default=None, help="Total token input size of base model.",)
parser.add_argument("--max_n_segments", type=int, default=None, help="Maximun number of segments to include from long input.",)
parser.add_argument("--curriculumLearning", default=False,  help="Whether or not to apply curriculum training.",)



# OTHER
parser.add_argument("--cache_dir", default="", type=str, help="Where do you want to store the pre-trained models downloaded from s3",)
parser.add_argument("--overwrite_cache", action="store_true", help="Overwrite the cached training and evaluation sets",)
parser.add_argument("--do_lower_case", action="store_true", help="Set this flag if you are using an uncased model.",)


args = parser.parse_known_args()[0]


# Read parameters from YAML file
if args.params:
    with open(args.params, 'r') as file:
        yaml_params = yaml.safe_load(file)
        for key, value in yaml_params['fineTuneModel'].items():
            parser.set_defaults(**{key: value})
        for key, value in yaml_params['modelParams'].items():
            parser.set_defaults(**{key: value})
        for key, value in yaml_params['RMT'].items():
            parser.set_defaults(**{key: value})

args = parser.parse_known_args()[0]

In [57]:
def load_data(args, tokenizer, test_run=False, split='train.fasta'):
    from Bio import SeqIO
    data = list(SeqIO.parse(os.path.join(args.data_dir, split), 'fasta'))
    labels = []
    seqs = []
    tr_ids = []
    for record in data:
        utr5, utr3 = record.seq.split(',')
        utr5 = tokenizer.encode(str(utr5).replace('U',"T"), add_special_tokens=True)
        utr3 = tokenizer.encode(str(utr3).replace('U',"T"), add_special_tokens=True) #, max_length=args.max_seq_length-len(utr5)+1, pad_to_max_length=False, truncation=True)
        s = utr5 + utr3[1:]
        if len(s) < 10 or len(s) > args.max_seq_length:
            continue
        # pad s to max_seq_length
        s = s + [tokenizer.pad_token_id]*(args.max_seq_length-len(s))
        if s not in seqs: # prevent duplicates
            seqs.append(s)
            labels.append(float(record.id))
            tr_ids.append(record.description.split(' ')[1])

    return torch.tensor(labels), torch.tensor(seqs), tr_ids

In [58]:
tokenizer = AutoTokenizer.from_pretrained('AIRI-Institute/gena-lm-bert-base-fly')

In [59]:
labels, seqs, tr_ids = load_data(args, tokenizer)

In [60]:
model_dir = 'output/ftModel/best_spearmanr'
model = GenaLMWithExtraFeatures.from_pretrained(model_dir) #, num_labels=1, id2label={0: "LABEL_0"})
model.eval()

/mnt/mr01-home01/m65338lb/worktrees/rnaDecay/gena_lm_extraFeatures/scripts/GenaLMWithExtraFeatures.py:130: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dic

GenaLMWithExtraFeatures(
  (model): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=3)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (pre_attention_ln): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (post_attention_ln): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
              (soft

In [61]:
import joblib
scaler_path = os.path.join(model_dir, 'scaler.joblib')
if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)

In [62]:
import pandas as pd
extraFeatures = pd.read_csv('output/data/codons/extraFeatures.csv', index_col=0)
extraFeatures.drop(['Decay Rate', 'Residuals'], axis=1, inplace=True)
scale1_extraFeatures = pd.DataFrame(scaler.transform(extraFeatures), columns=extraFeatures.columns, index=extraFeatures.index)

In [63]:
scale1_extraFeatures

,5' UTR Length,CDS Length,3' UTR Length,5' UTR GC Content,CDS GC Content,3' UTR GC Content,TTT,TTC,TTA,TTG,...,CGT,CGC,CGA,CGG,AGA,AGG,GGT,GGC,GGA,GGG
FBtr0310660,-0.526057,-0.654695,-0.959950,0.614858,1.571407,0.084789,-1.231010,-0.343071,0.024266,-0.363964,...,-0.843729,-1.368335,-0.649157,-1.026522,4.323254,4.309661,-0.809509,-1.112170,-1.088520,-0.222439
FBtr0084659,-0.709332,-0.672587,-0.893719,1.155092,-0.129178,-0.894421,0.695427,1.214091,-0.365567,0.856226,...,0.207110,-0.450008,-0.328084,0.536601,-0.614024,-0.631363,1.102631,1.210073,-0.121474,-0.740780
FBtr0343763,1.891987,1.196279,0.476881,-0.504826,0.734564,-0.334429,-0.331406,-1.156338,-0.489912,0.310496,...,0.196469,0.804741,-0.458705,-0.457158,-0.511776,-0.438789,2.074788,0.795593,0.395779,-0.648353
FBtr0079234,0.085844,-1.036926,0.031666,1.617418,0.745280,-0.765020,2.336467,-0.866955,0.372444,0.291664,...,-1.179921,-1.589238,-1.235974,-1.327785,0.547749,1.612111,0.556887,-0.070308,-0.888793,-1.292759
FBtr0074180,-0.283662,1.487425,-0.687669,-1.025487,-0.618300,-1.513242,0.309298,-0.989299,-0.001399,-0.249243,...,0.628262,-0.115979,4.571351,-0.420406,-0.496695,-0.529281,1.452534,0.772851,1.628622,0.894763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FBtr0332144,0.774602,0.086996,0.563349,0.472068,-0.140012,0.116731,-0.680824,-0.268884,-0.425016,1.469053,...,-0.734271,-0.613154,-0.069163,-0.129740,-0.541294,-0.473031,-0.894697,-0.683465,-0.734150,-0.346895
FBtr0089259,-0.901475,-0.615659,3.635701,-0.352617,0.170672,-0.413135,-0.355357,0.270717,-0.403306,-0.813771,...,-0.549452,-0.967840,-0.135501,-0.480337,0.058310,-0.651534,-1.542129,-0.099408,-0.251365,-0.790959
FBtr0089344,-0.070827,-0.093548,-0.516574,0.565859,0.297147,-0.742351,-0.116805,-0.942638,0.042508,-0.910905,...,0.882944,0.105084,1.314521,1.136936,0.395232,1.676713,-0.980211,-0.147385,-0.133617,0.622750
FBtr0089362,-0.070827,-0.485538,-0.569926,0.565859,0.358818,-0.788526,0.460664,-1.189000,-0.468233,-0.997162,...,1.430084,0.640223,2.180818,1.011050,-0.293188,1.317859,-0.688981,-0.080634,0.062887,0.784582


In [64]:
s = StandardScaler()
s.fit(extraFeatures[extraFeatures.index.isin(tr_ids)])
scale2_extraFeatures = pd.DataFrame(s.transform(extraFeatures), columns=extraFeatures.columns, index=extraFeatures.index)
scale2_extraFeatures

,5' UTR Length,CDS Length,3' UTR Length,5' UTR GC Content,CDS GC Content,3' UTR GC Content,TTT,TTC,TTA,TTG,...,CGT,CGC,CGA,CGG,AGA,AGG,GGT,GGC,GGA,GGG
FBtr0310660,-0.526057,-0.654695,-0.959950,0.614858,1.571407,0.084789,-1.231010,-0.343071,0.024266,-0.363964,...,-0.843729,-1.368335,-0.649157,-1.026522,4.323254,4.309661,-0.809509,-1.112170,-1.088520,-0.222439
FBtr0084659,-0.709332,-0.672587,-0.893719,1.155092,-0.129178,-0.894421,0.695427,1.214091,-0.365567,0.856226,...,0.207110,-0.450008,-0.328084,0.536601,-0.614024,-0.631363,1.102631,1.210073,-0.121474,-0.740780
FBtr0343763,1.891987,1.196279,0.476881,-0.504826,0.734564,-0.334429,-0.331406,-1.156338,-0.489912,0.310496,...,0.196469,0.804741,-0.458705,-0.457158,-0.511776,-0.438789,2.074788,0.795593,0.395779,-0.648353
FBtr0079234,0.085844,-1.036926,0.031666,1.617418,0.745280,-0.765020,2.336467,-0.866955,0.372444,0.291664,...,-1.179921,-1.589238,-1.235974,-1.327785,0.547749,1.612111,0.556887,-0.070308,-0.888793,-1.292759
FBtr0074180,-0.283662,1.487425,-0.687669,-1.025487,-0.618300,-1.513242,0.309298,-0.989299,-0.001399,-0.249243,...,0.628262,-0.115979,4.571351,-0.420406,-0.496695,-0.529281,1.452534,0.772851,1.628622,0.894763
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
FBtr0332144,0.774602,0.086996,0.563349,0.472068,-0.140012,0.116731,-0.680824,-0.268884,-0.425016,1.469053,...,-0.734271,-0.613154,-0.069163,-0.129740,-0.541294,-0.473031,-0.894697,-0.683465,-0.734150,-0.346895
FBtr0089259,-0.901475,-0.615659,3.635701,-0.352617,0.170672,-0.413135,-0.355357,0.270717,-0.403306,-0.813771,...,-0.549452,-0.967840,-0.135501,-0.480337,0.058310,-0.651534,-1.542129,-0.099408,-0.251365,-0.790959
FBtr0089344,-0.070827,-0.093548,-0.516574,0.565859,0.297147,-0.742351,-0.116805,-0.942638,0.042508,-0.910905,...,0.882944,0.105084,1.314521,1.136936,0.395232,1.676713,-0.980211,-0.147385,-0.133617,0.622750
FBtr0089362,-0.070827,-0.485538,-0.569926,0.565859,0.358818,-0.788526,0.460664,-1.189000,-0.468233,-0.997162,...,1.430084,0.640223,2.180818,1.011050,-0.293188,1.317859,-0.688981,-0.080634,0.062887,0.784582


In [65]:
args.data_dir

'output/data/decay'

In [66]:
seqs[0].tolist()

[1,
 315,
 140,
 628,
 12079,
 208,
 3203,
 148,
 315,
 3202,
 2992,
 103,
 804,
 860,
 6008,
 50,
 15240,
 2393,
 176,
 999,
 1174,
 55,
 2335,
 2116,
 436,
 562,
 2485,
 509,
 1151,
 214,
 1130,
 425,
 22343,
 130,
 226,
 9705,
 304,
 232,
 364,
 21048,
 76,
 319,
 47,
 317,
 982,
 594,
 16946,
 5603,
 2690,
 8196,
 235,
 424,
 148,
 10084,
 1054,
 6645,
 148,
 18,
 2,
 94,
 7486,
 1596,
 1110,
 549,
 33,
 489,
 1004,
 4492,
 86,
 2809,
 409,
 30,
 632,
 6868,
 125,
 23716,
 117,
 586,
 24,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3

In [73]:
from Bio import SeqIO
z = list(SeqIO.parse(os.path.join(args.data_dir, 'train.fasta'), 'fasta'))
seq = z[0]
del z


In [72]:
seq

SeqRecord(seq=Seq('UUCGUUUAGUGUCUGAGGCAGAGCUCUGCAUGCUUGUCGUUCGAAGCGCUUUUA...CAC'), id='4.856488661779951', name='4.856488661779951', description='4.856488661779951 FBtr0078968 FBgn0010225 NP_524865.2|DROME10448|33436|[Dmela] 33436', dbxrefs=[])

In [74]:
utr5, utr3 = str(seq.seq).replace('U','T').split(',')
utr5 = tokenizer.encode(str(utr5), add_special_tokens=True)
utr3 = tokenizer.encode(str(utr3), add_special_tokens=True) #, max_length=args.max_seq_length-len(utr5)+1, pad_to_max_length=False, truncation=True)
s = utr5 + utr3[1:]
if len(s) > args.max_seq_length:
    print(f"Skipping sequence {seq.id} due to length {len(s)}")
else:
    am = [1] * len(s) + [0] * (args.max_seq_length - len(s))
    s = s + [tokenizer.pad_token_id] * (args.max_seq_length - len(s))


s


[1,
 315,
 140,
 628,
 12079,
 208,
 3203,
 148,
 315,
 3202,
 2992,
 103,
 804,
 860,
 6008,
 50,
 15240,
 2393,
 176,
 999,
 1174,
 55,
 2335,
 2116,
 436,
 562,
 2485,
 509,
 1151,
 214,
 1130,
 425,
 22343,
 130,
 226,
 9705,
 304,
 232,
 364,
 21048,
 76,
 319,
 47,
 317,
 982,
 594,
 16946,
 5603,
 2690,
 8196,
 235,
 424,
 148,
 10084,
 1054,
 6645,
 148,
 18,
 2,
 94,
 7486,
 1596,
 1110,
 549,
 33,
 489,
 1004,
 4492,
 86,
 2809,
 409,
 30,
 632,
 6868,
 125,
 23716,
 117,
 586,
 24,
 2,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3,
 3

In [94]:
tmp_extraFeatures = scale1_extraFeatures.loc[seq.description.split()[1]].to_numpy()
o = model(input_ids=torch.tensor([s]), attention_mask=torch.ones_like(torch.tensor([s])), extra_features=torch.tensor([tmp_extraFeatures], dtype=torch.float32))
o

/mnt/mr01-home01/m65338lb/.local/share/mamba/envs/inseq/lib/python3.10/site-packages/transformers/modeling_utils.py:1044: FutureWarning: The `device` argument is deprecated and will be removed in v5 of Transformers.
  warnings.warn(


SequenceClassifierOutput(loss=None, logits=tensor([[3.5401]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [93]:
seqs[0].shape

torch.Size([512])

In [ ]:
train_dataset = TensorDataset(seqs, torch.ones_like(seqs),torch.zeros_like(seqs),labels, tr_ids_index) #load_and_cache_examples(args, args.task_name, tokenizer, evaluate=False)
inputs = {"input_ids": batch[0].to(args.device), "attention_mask": batch[1].to(args.device), "labels": batch[3].to(args.device), "extra_features": tmp_extraFeatures}